# Part 9, notebook 5 of 5: the review queue and what goes wrong

Companion to *Human in the Loop*. The five notebooks follow the post in order. Each explains the idea with a daily-life picture, shows the real source, and runs it. **Cells marked "Your turn" are for you to change and rerun.**

| # | Notebook | Post section | Needs the model server? |
|---|---|---|---|
| 09a | `09a_where_a_human_belongs` | Where a person belongs in the pipeline | no |
| 09b | `09b_pause_and_resume` | Pause, ask, resume: `interrupt()` | no |
| 09c | `09c_who_to_ask` | Which cases to send: the review budget | no (reads the saved Part 8 runs) |
| 09d | `09d_does_the_human_help` | Does the human step help, and what does it cost? | no (reads the saved Part 7 runs) |
| 09e | `09e_queue_and_failures` | The review queue and what goes wrong | one cell uses the model server |

**The picture.** The counter at a deli with ticket numbers. The kitchen (the pipeline) drops orders on the counter, the clerk (the reviewer) takes the oldest ticket, and the manager looks at the board to see how many are waiting. What the board must never do is quietly mark an unanswered ticket as served.

`hitl_queue.py` is that board: one SQLite file with LangGraph's saved states plus one table of tickets. In this notebook we fill it, answer some, leave some, and break it on purpose.

In [1]:
import sys, json, os
from pathlib import Path
os.environ.setdefault("AGENTIC_ZERO_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves (only 09e calls it)
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))                # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
%matplotlib inline
import tempfile, time
import hitl_eval as ev, hitl_graph as h
from hitl_queue import ReviewQueue

## 1. Fill the queue

40 images go in. Images that need no person finish at once; the others wait with a ticket. `clock` is injected so we can move time forward later without waiting.

In [2]:
db = tempfile.mktemp(suffix=".db")
now = [1_000_000.0]
q = ReviewQueue(db, classifier="band", policy="all_nonpass", clock=lambda: now[0])
for i, im in enumerate(ev.TEST[:40]):
    now[0] += 60                                  # one image per minute
    q.submit(im)
print("counts:", q.counts())
for p in q.pending()[:3]:
    print(" waiting:", p["thread_id"], f"{p['waiting_s']/60:.0f} min", "| verdict", p["payload"]["pipeline_verdict"])

counts: {'done': 23, 'waiting': 17}
 waiting: insp-test_00002 37 min | verdict review
 waiting: insp-test_00005 34 min | verdict review
 waiting: insp-test_00006 33 min | verdict review


## 2. A reviewer works through some of them

The simulated reviewer from notebook 09d answers the first 10. Everything it does goes through the public `answer()` call, the same one a web form would use.

In [3]:
rev = ev.Reviewer(error_rate=0.05, seed=1)
for p in q.pending()[:10]:
    now[0] += 30
    q.answer(p["thread_id"], rev.decide(p["payload"]))
print("counts:", q.counts())
print("rows closed by a person:", q.conn.execute("SELECT thread_id, verdict, reviewer FROM review_queue WHERE reviewer IS NOT NULL LIMIT 3").fetchall())

counts: {'done': 33, 'waiting': 7}
rows closed by a person: [('insp-test_00002', 'fail', 'ana'), ('insp-test_00005', 'fail', 'ana'), ('insp-test_00006', 'fail', 'ana')]


## 3. Nobody answers the rest

Two hours pass. `expire()` does **not** pass these images. It marks them `held`: the part stays blocked and someone has to be told. A silent timeout that turns into a pass is the worst possible default for an inspection.

In [4]:
now[0] += 2 * 3600
held = q.expire(older_than_s=3600)
print("held:", len(held), "| counts:", q.counts())
print("verdicts stored for held items:", set(v for (v,) in q.conn.execute("SELECT verdict FROM review_queue WHERE status='held'")))

held: 7 | counts: {'done': 33, 'held': 7}
verdicts stored for held items: {'hold'}


## 4. Things that must be refused

In [5]:
tid = q.conn.execute("SELECT thread_id FROM review_queue WHERE status='done' AND reviewer IS NOT NULL LIMIT 1").fetchone()[0]
for label, fn in [("answer a closed item again", lambda: q.answer(tid, {"action": "reject", "reviewer": "bo"})),
                  ("answer an item that was held", lambda: q.answer(held[0], {"action": "approve", "reviewer": "bo"})),
                  ("answer a ticket that does not exist", lambda: q.answer("insp-nope", {"action": "approve", "reviewer": "bo"}))]:
    try:
        fn(); print(f"{label}: ACCEPTED (bad)")
    except Exception as e:
        print(f"{label}: {type(e).__name__}: {e}")
print("submit the same image twice:", q.submit(ev.TEST[0]))

answer a closed item again: RuntimeError: insp-test_00002 is 'done', not waiting: answer refused
answer an item that was held: RuntimeError: insp-test_00027 is 'held', not waiting: answer refused
answer a ticket that does not exist: KeyError: 'no such review item: insp-nope'
submit the same image twice: {'status': 'done', 'thread_id': 'insp-test_00000', 'note': 'already submitted: not run again'}


## 5. The same graph with a language model and a written note

Everything above used the rule classifier and template notes so it runs offline. This cell uses the real Part 7 setup: the local model classifies the boxes and writes the note. **It needs the vLLM server** (about 15 to 30 s per image). If the server is down the cell prints the error instead of failing the notebook.

In [6]:
import traceback
try:
    from langgraph.checkpoint.memory import MemorySaver
    gl = h.build_hitl("llm", "all_nonpass", prose=True, checkpointer=MemorySaver())
    for im in ("test_00002", "test_00006"):
        t = time.time(); r = h.start(gl, im, im)
        print(im, "->", r["status"], f"{time.time()-t:.1f} s", "| pipeline verdict:", r.get("payload", {}).get("pipeline_verdict"))
        if r["status"] == "waiting":
            t = time.time(); r = h.resume(gl, im, ev.Reviewer().decide(r["payload"]))
            s = r["state"]
            print("   after the person:", s["verdict"], "| note by", s["report"]["summary_source"], f"({time.time()-t:.1f} s):", s["report"]["summary"][:160])
except Exception as e:
    print("model server not reachable:", type(e).__name__, str(e)[:120])

test_00002 -> waiting 12.0 s | pipeline verdict: escalate


   after the person: fail | note by llm (3.6 s): Image test_00002 received a fail verdict because the analysis selected the 'edit' action. Candidate 0 was kept as a critical defect with 0.65 confidence, while 


test_00006 -> waiting 24.1 s | pipeline verdict: review


   after the person: fail | note by llm (5.1 s): The inspection of image test_00006 resulted in a fail verdict because the analyst chose to edit. Two candidates were kept: candidate 0, a major defect with 1.0 


Look at the note the model wrote. It says things like *"the analyst chose to approve"*. The facts are right (the ids and numbers come from the record, and the prose check in Part 7 rejects invented ids), but the model has no idea **why** the person decided. The reviewer's reason is only in the audit trail. Do not let a generated note stand in for the reviewer's own words.

## 6. The test suite

`test_hitl_paths.py` walks every path above plus the ones that are annoying to do by hand (seven kinds of malformed answer, a rescued box, a second process resuming, graph versus simulation for four reviewers). It is offline and takes seconds.

In [7]:
import subprocess, sys
r = subprocess.run([sys.executable, "test_hitl_paths.py"], cwd=str(Path("..").resolve()), capture_output=True, text=True)
print(r.stdout[-2400:] or r.stderr[-1500:])

ok  approve keeps the pipeline's boxes, verdict fail, decided_by human
ok  reject clears everything, verdict pass, the note is in the reason
ok  edit keeps exactly the given ids
ok  7 malformed answers are each refused and the question is asked again
ok  the audit trail keeps every refused answer and the final decision
ok  edit can add a rejected box; it is flagged added_by_human
ok  policy none never pauses
ok  escalate_only pauses less than all_nonpass
ok  the audit policy pauses on some images that the pipeline passed
ok  without a human a 'review' verdict becomes fail by itself (decided_by auto-fail)
ok  each resume starts the review node again from its first line
ok  a brand-new process finishes a review started in this one
ok  a second answer to a closed item is refused
ok  an unknown item raises KeyError
ok  submitting the same image twice does not run it twice
ok  expire marks unanswered items 'held', never 'pass'
ok  graph == simulation (reviewer error 0.0, scope all, rubber_s